In [ ]:
# 1. Partitions in Hive
# # 2. Types of Partitions
#       a) static -  Provide partition name and the inserted data
#       b) Dynamic - Based on distinct column values distinct partitions will be created


In [ ]:
# What is the partition?
#    When you write sql query select * from tablename where filter_condition system will scan all the rows in the table and filter the data
#    while partitions will be created based on unique column values into seperate folder and makes it easy to go through folder names instead of entire data

In [1]:
%%sql
show CATALOGS

StatementMeta(, 0105d970-1dc8-4ddb-8b32-1fea32e19f44, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
%%sql
show DATABASES

StatementMeta(, 0105d970-1dc8-4ddb-8b32-1fea32e19f44, 3, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 1 fields>

In [3]:
%%sql

USE db_hymaa_test

StatementMeta(, 0105d970-1dc8-4ddb-8b32-1fea32e19f44, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql

show TABLES

StatementMeta(, 0105d970-1dc8-4ddb-8b32-1fea32e19f44, 5, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 3 fields>

In [1]:
#IN PYTHON
DF=spark.sql("DESCRIBE DETAIL db_hymaa_test.dept_dtls")
display(DF)

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45c978c0-e555-42d3-8133-9a3cab832a21)

In [2]:
%%sql

--IN SQL

DESCRIBE DETAIL db_hymaa_test.dept_dtls

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 15 fields>

In [5]:
%%sql

use db_hymaa_test

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [27]:
#Create dynamic partition table by single column - state

from pyspark.sql.types import StructType,StructField,StringType,IntegerType

population_schema=StructType([
    StructField('pop_id',StringType(),True),
    StructField('pop_name',StringType(),True),
    StructField('pop_salary',IntegerType(),True),
    StructField('pop_gender',StringType(),True),
    StructField('pop_age',IntegerType(),True),
    StructField('pop_state',StringType(),True)
])

df=spark.read.csv(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/Popuation',
    header=False,
    schema=population_schema)

display(df)

df.write.format('delta').mode('append').partitionBy('pop_state').saveAsTable('population')

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 000c39be-a2bb-4a6d-b490-2fa854b58ed8)

In [28]:
%%sql

DESC formatted population

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 30, Finished, Available, Finished, False)

<Spark SQL result set with 16 rows and 3 fields>

In [36]:
%%sql 
show PARTITIONS population

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 38, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 1 fields>

In [34]:
%%sql
select * from population

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 36, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 6 fields>

In [32]:
#Create static partition table by single column - state

df1=df.filter(df.pop_state=='Tamilnadu')
df1.write.format('delta').mode('append').partitionBy('pop_state').saveAsTable('population_static')

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 34, Finished, Available, Finished, False)

In [33]:
%%sql
select * from population_static

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 35, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 6 fields>

In [30]:
%%sql

create table if not EXISTS population_static
(
    pop_id string,
    pop_name string,
    pop_salary INT,
    pop_gender string,
    pop_age int,
    pop_state string

)
using DELTA
LOCATION 'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Tables/db_hymaa_test/population'
PARTITIONED BY (pop_state)

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 32, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [35]:
%%sql 
show PARTITIONS population_static

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 37, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [37]:
#Create dynamic partition table by multiple columns - state and gender

population_schema=StructType([
    StructField('pop_id',StringType(),True),
    StructField('pop_name',StringType(),True),
    StructField('pop_salary',IntegerType(),True),
    StructField('pop_gender',StringType(),True),
    StructField('pop_age',IntegerType(),True),
    StructField('pop_state',StringType(),True)
])

df=spark.read.csv(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/Popuation',
    header=False,
    schema=population_schema)

display(df)

df.write.format('delta').mode('append').partitionBy(['pop_state','pop_gender']).saveAsTable('population_dynamic_multi')

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8d099ef-f2b8-41de-b8dc-4be2838c625a)

In [38]:
%%sql
show partitions population_dynamic_multi

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 40, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 1 fields>

In [54]:
#Create static partition table by multiple columns - state and gender

df1=df.filter((df.pop_state=='Tamilnadu') & (df.pop_gender=='M'))
df1.write.format('delta').mode('append').partitionBy('pop_state','pop_gender').saveAsTable('population_static_multi')

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 56, Finished, Available, Finished, False)

In [49]:
%%sql
select * from population_static_multi

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 51, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 6 fields>

In [50]:

 %%sql

--#Register the static table in fabric sql

create table if not EXISTS population_static_multi
(
    pop_id string,
    pop_name string,
    pop_salary INT,
    pop_gender string,
    pop_age int,
    pop_state string

)
using DELTA
LOCATION 'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Tables/db_hymaa_test/population'
PARTITIONED BY (pop_state,pop_gender)

StatementMeta(, 33ca1d1d-cd60-45fa-978d-65b057b2c699, 52, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

%%sql
--#Databricks Syntax for partition


    --#Create table using csv location

    create table if not exists population(
        pop_id string,
        pop_name string,
        pop_salary int,
        pop_gender string,
        pop_age int,
        pop_state string
     )
     ROW FORMAT DELIMITED
     FIELDS TERMINATED BY ','
     LOCATION 'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Tables/db_hymaa_test/population';

    --#Create static partition table using csv location

    create table if not exists population_static(
        pop_id string,
        pop_name string,
        pop_salary int,
        pop_gender string,
        pop_age int
     )
     PARTITIONED BY (pop_state string)
     ROW FORMAT DELIMITED
     FIELDS TERMINATED BY ',';

     LOAD DATA INPATH 'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Tables/db_hymaa_test/population' INTO TABLE population_static
     PARTITION(pop_state='Tamilnadu');

    
    --Create table with partition using existing table

    create table if not exists population_multi(
        p_pop_id string,
        p_pop_name string,
        p_pop_salary int,
        p_pop_age int
      )
      PARTITIONED BY (p_pop_state string,p_pop_gender string)
      ROW FORMAT DELIMITED
      FIELDS TERMINATED BY ',';


     INSERT INTO population_multi PARTITION(p_pop_state,p_pop_gender)
     SELECT 
     pop_id as 'p_pop_id',
     pop_name as 'p_pop_name',
     pop_salalry as 'p_pop_salary',
     pop_age as 'p_pop_age',
     pop_state as 'p_pop_state',
     pop_gender as 'p_pop_gender'
     FROM population
